In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 7 — Local Research Paper Explainer
## Explain Research Papers in Plain English

**Stack:** Ollama · LlamaIndex · Jupyter

In [ ]:
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Setup and Sample Paper

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

sample_paper = {
    "title": "Attention Is All You Need",
    "abstract": """The dominant sequence transduction models are based on complex
recurrent or convolutional neural networks that include an encoder and a decoder.
The best performing models also connect the encoder and decoder through an
attention mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to be
superior in quality while being more parallelizable and requiring significantly
less time to train.""",
    "sections": {
        "Introduction": """Recurrent neural networks, long short-term memory and gated
recurrent neural networks have been established as state of the art approaches in
sequence modeling and transduction problems. The Transformer is the first
transduction model relying entirely on self-attention to compute representations.""",
        "Self-Attention": """An attention function maps a query and a set of key-value
pairs to an output. We call our particular attention Scaled Dot-Product Attention.
Multi-Head Attention allows the model to jointly attend to information from
different representation subspaces at different positions.""",
        "Architecture": """The encoder maps an input sequence to a sequence of continuous
representations. The decoder generates an output sequence of symbols one at a time.
The Transformer follows this overall architecture using stacked self-attention and
point-wise, fully connected layers for both encoder and decoder."""
    }
}
print(f"Loaded paper: {sample_paper['title']}")

## Step 2 — Section-by-Section Explanation

In [ ]:
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", """You explain research paper sections in plain English.
Target audience: smart undergraduate who hasn't studied this field.
- Use analogies and everyday examples
- Define technical terms when first used
- Keep explanations under 200 words per section
- Highlight the key insight of each section"""),
    ("human", """Paper: {title}

Section: {section_name}
Content: {section_content}

Explain this section in plain English.""")
])

explain_chain = explain_prompt | llm | StrOutputParser()

print(f"# {sample_paper['title']}\n")
print("## Abstract (Plain English)")
abstract_explained = explain_chain.invoke({
    "title": sample_paper["title"],
    "section_name": "Abstract",
    "section_content": sample_paper["abstract"],
})
print(abstract_explained)

for section_name, content in sample_paper["sections"].items():
    print(f"\n## {section_name} (Plain English)")
    explanation = explain_chain.invoke({
        "title": sample_paper["title"],
        "section_name": section_name,
        "section_content": content,
    })
    print(explanation)

## Step 3 — Key Takeaways Extractor

In [ ]:
from pydantic import BaseModel, Field

class PaperTakeaways(BaseModel):
    one_sentence_summary: str
    key_contribution: str
    main_technique: str
    practical_applications: list[str]
    limitations: list[str]
    prerequisites_to_understand: list[str]

takeaway_llm = llm.with_structured_output(PaperTakeaways)

full_text = f"Title: {sample_paper['title']}\nAbstract: {sample_paper['abstract']}"
for name, content in sample_paper["sections"].items():
    full_text += f"\n\n{name}: {content}"

takeaways = takeaway_llm.invoke(f"Extract key takeaways from this paper:\n\n{full_text}")
print(f"One-Sentence Summary: {takeaways.one_sentence_summary}")
print(f"Key Contribution: {takeaways.key_contribution}")
print(f"Main Technique: {takeaways.main_technique}")
print(f"\nApplications:")
for a in takeaways.practical_applications:
    print(f"  • {a}")
print(f"\nLimitations:")
for l in takeaways.limitations:
    print(f"  • {l}")

## Step 4 — Generate Quiz Questions

In [ ]:
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate 5 quiz questions about this research paper. Include "
     "multiple choice (4 options each) and short answer questions. Provide the "
     "correct answers."),
    ("human", "{paper_text}")
])

quiz_chain = quiz_prompt | llm | StrOutputParser()
quiz = quiz_chain.invoke({"paper_text": full_text})
print("Quiz Questions:")
print(quiz)

## What You Learned
- **Section-by-section explanation** with audience-aware prompts
- **Structured takeaway extraction** with Pydantic
- **Quiz generation** from academic content